# Cattle Re-ID — Kaggle Runner (paper replication, Phases 0→3)

Orchestrator to run the **paper replication (VGG16_BN)** on **Kaggle with GPU**. Logic lives in `src/` and `scripts/`; this notebook only orchestrates (see `CLAUDE.md`). The ResNet-50 backbone (Phase 4, for domain adaptation) is separate and not run here.

**Before running — configure the notebook:**
1. *Settings → Accelerator → **GPU T4 ×2***. **Do NOT use P100**: the current Kaggle image ships a PyTorch that dropped Pascal/sm_60 support → `CUDA error: no kernel image is available`. T4 is sm_75, which is supported. (The paper used P100; see `DEVIATIONS.md`.)
2. *Settings → Internet → **ON*** (required for `git clone`).
3. *Add Input* → attach **two** datasets:
   - The **images** dataset (contains `BeefCattle_Muzzle_Individualized/`).
   - The **ImageNet weights** dataset (`vgg16_bn-6c64b313.pth`, without renaming).

`config.py` auto-detects both in `/kaggle/input/` (recursive search, works even if the mount lands at `/kaggle/input/datasets/<user>/<slug>/...`). Everything written to `/kaggle/working/` persists on **Save Version**.

> **For the long sweep use "Save & Run All" (commit)**: runs in the background for up to ~12 h without staying connected. The interactive session times out on idle.

## 0. Verify GPU

In [ ]:
!nvidia-smi
import torch
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU: Settings -> Accelerator -> GPU T4 x2 (not P100)'
# Sanity: GPU must be compatible with the image's PyTorch (T4 = sm_75 OK; P100 = sm_60 NO).
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    cap = torch.cuda.get_device_capability(0)
    print(f'GPU: {name} | capability sm_{cap[0]}{cap[1]}')
    if cap[0] < 7:
        print('WARNING: this GPU (sm_<70) may not be compatible with Kaggle\'s PyTorch. '
              'If the forward raises "no kernel image", switch to GPU T4 x2.')

## 1. Get the code (git clone / pull)

In [ ]:
import os

REPO_URL = 'https://github.com/benjaminvitale/tp-final-vision2-Pujol-Vitale.git'
REPO_DIR = '/kaggle/working/tp-final-vision2-Pujol-Vitale'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull --ff-only

%cd {REPO_DIR}
!git log --oneline -1

## 2. Verify paths (DATA_DIR + PRETRAINED_DIR)

`config.py` resolves the dataset and weights from `/kaggle/input/`. **Do not proceed** if `DATA_DIR` does not exist or `PRETRAINED_DIR` is `None` (without the `.pth` files, models will try to download them from the internet).

In [ ]:
import config, importlib
importlib.reload(config)

print('DATA_DIR       :', config.DATA_DIR, '| exists:', config.DATA_DIR.is_dir())
print('PRETRAINED_DIR :', config.PRETRAINED_DIR)

assert config.DATA_DIR.is_dir(), 'DATA_DIR not found: attach the image dataset (Add Input).'
if config.PRETRAINED_DIR is None:
    print('WARNING: ImageNet .pth files not found. Models will try to download them '
          '(Internet must be ON). Recommended: attach the weights dataset.')

## 3. Phase 0 — Data inspection (ALWAYS first)

Sanity check against the paper: **268 classes, 4923 images, 4–70 per class**. Do not proceed if it does not match.

In [ ]:
!python scripts/00_inspect_data.py

## 4. Splits (already committed — DO NOT regenerate)

Splits live in `outputs/splits/` with paths relative to `DATA_DIR`, so the same split works on Kaggle and locally. This cell only **verifies** them. Run `01_make_splits.py` only if you intentionally want to regenerate (breaks reproducibility with the committed splits).

In [ ]:
from src.utils import load_json
for name in ('train', 'val', 'test'):
    n = len(load_json(config.SPLITS_DIR / f'{name}.json'))
    print(f'{name:5}: {n} images')
label_map = load_json(config.SPLITS_DIR / 'label_map.json')
print('classes in label_map:', len(label_map))

## 4b. Precompute augmentation (synthetic images to disk)

PAPER: augmentation is a **pre-processing step** that creates synthetic images and enlarges the dataset. Here we generate them **once** (originals + copies for classes with few images, up to `min(20, 2·N_i)`) and they are fixed in `outputs/aug_cache/`. The `ce_aug` variant uses them automatically; `ce`/`wce` do not.

> Does not speed up training (the bottleneck is the GPU forward pass), but fixes and makes the synthetic set reproducible. The **smoke** test uses online fallback, so this cell is only for the real sweep.

In [ ]:
# Precompute the synthetic augmentation set to disk (used only by the ce_aug variant).
# Required before the real sweep; smoke uses online fallback. ~25 MB, ~1 min.
!python scripts/04_precompute_aug.py

## 5. GPU smoke test (validate BEFORE burning quota)

Confirms that paths + GPU + pipeline resolve end-to-end. Uses a small subset, 2 epochs, and `pretrained=False` (no weights download). Accuracy will be ~0: expected — this validates the pipeline, not the science. ~1–2 min.

In [ ]:
!python scripts/02_train_vgg.py --smoke

## 6. Phase 3 — VGG16_BN replication (the paper deliverable)

4 variants (`ce`, `ce_aug`, `wce`, `wce_aug`) × **3 seeds**, 50 epochs. **This is the long sweep** → run with *Save & Run All* (background).

**Budget (measured on T4):** `ce` ~44 min, `ce_aug` ~65 min (1.47× dataset due to additive augmentation), `wce` ~44 min, `wce_aug` ~65 min → 4 var × 3 seeds ≈ **9.5 h**. Fits within the 12 h limit.

Success validation: best variant ~96–98%+, and `ce_aug`/`wce` improve accuracy on 4-image classes vs `ce`. Result in `outputs/results/02_vgg_summary.json`.

In [ ]:
# 3 seeds to fit within the 12 h T4 budget (see DEVIATIONS.md D2).
# For all 5 from the plan: --seeds 0 1 2 3 4 (may not fit in a single commit; split it).
!python scripts/02_train_vgg.py --seeds 0 1 2

## 7. Results and checkpoints (persisted on Save Version)

In [ ]:
import json

print('=== outputs/results/ ===')
!ls -lh outputs/results/
print('\n=== outputs/checkpoints/ ===')
!ls -lh outputs/checkpoints/

p = config.RESULTS_DIR / '02_vgg_summary.json'
if p.exists():
    print('\n=== 02_vgg_summary.json (replication summary) ===')
    data = json.loads(p.read_text())
    print(json.dumps(data.get('summary', data), indent=2, ensure_ascii=False)[:2500])